# Experiment 8: Feature Extraction from Time Series using CNN

## Step 1
Import the required libraries.

In [1]:
import os
import gc
import glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import kagglehub

torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


## Step 2
Load the dataset and treat each image as a 1D signal of 784 values.

In [2]:
path = kagglehub.dataset_download("shayanfazeli/heartbeat")


def find_csv(name):
    return glob.glob(os.path.join(path, "**", name), recursive=True)[0]


train_data = pd.read_csv(find_csv("mitbih_train.csv"), header=None).sample(frac=1.0, random_state=42).values.astype(np.float32)
test_data = pd.read_csv(find_csv("mitbih_test.csv"), header=None).values.astype(np.float32)

Xall = torch.tensor(train_data[:, :187]).unsqueeze(1)
yall = torch.tensor(train_data[:, 187]).long()
Xte = torch.tensor(test_data[:, :187]).unsqueeze(1)
yte = torch.tensor(test_data[:, 187]).long()

n_val = int(0.1 * len(Xall))
Xt, yt = Xall[:-n_val], yall[:-n_val]
Xv, yv = Xall[-n_val:], yall[-n_val:]

n_classes = 5
counts = torch.bincount(yt, minlength=n_classes).float()
class_weights = counts.sum() / (n_classes * counts.clamp(min=1))
batch_size = 256


def make_loader(X, y, bs, shuffle):
    return DataLoader(TensorDataset(X, y), batch_size=bs, shuffle=shuffle)


print(Xt.shape, Xv.shape, Xte.shape)
print("training samples per class:", counts.int().tolist())

torch.Size([78799, 1, 187]) torch.Size([8755, 1, 187]) torch.Size([21892, 1, 187])
training samples per class: [65203, 2013, 5187, 587, 5809]


## Step 3
Define the 1D convolutional layers, pooling layers and the output layer.

In [3]:
class Extractor(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(1, 32, 7, padding=3), nn.BatchNorm1d(32), nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(32, 64, 5, padding=2), nn.BatchNorm1d(64), nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 128, 3, padding=1), nn.BatchNorm1d(128), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(128, 64), nn.ReLU(),
        )

    def forward(self, x):
        return self.net(x)


class SeriesNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.extractor = Extractor()
        self.head = nn.Linear(64, n_classes)

    def forward(self, x):
        return self.head(self.extractor(x))


epochs = 10
use_amp = device.type == "cuda"
model = SeriesNet().to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
opt = torch.optim.Adam(model.parameters(), lr=2e-3)
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
print(f"parameters: {sum(p.numel() for p in model.parameters())}")

parameters: 44293


## Step 4
Train the CNN for 5 epochs, then remove the output layer and extract the 64 features.

In [4]:
def free_memory():
    if device.type == "cuda":
        torch.cuda.empty_cache()


def is_oom(e):
    return isinstance(e, torch.cuda.OutOfMemoryError) or "out of memory" in str(e).lower()


def safe_call(fn, bs, min_bs=8):
    while True:
        try:
            return fn(bs), bs
        except RuntimeError as e:
            if not is_oom(e):
                raise
        model.zero_grad(set_to_none=True)
        gc.collect()
        free_memory()
        if bs <= min_bs:
            raise RuntimeError("out of memory even at the smallest batch size")
        bs = max(min_bs, bs // 2)
        print(f"out of memory, retrying with batch size {bs}")


def train_one_epoch(bs):
    model.train()
    tot = 0.0
    n = 0
    for x, y in make_loader(Xt, yt, bs, True):
        x, y = x.to(device), y.to(device)
        opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, enabled=use_amp):
            out = model(x)
        loss = criterion(out.float(), y)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        tot += loss.item() * len(y)
        n += len(y)
    return tot / n


def predict(m, X, bs):
    m.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(X), bs):
            with torch.autocast(device_type=device.type, enabled=use_amp):
                out.append(m(X[i:i + bs].to(device)).argmax(1).cpu())
    return torch.cat(out)


def extract(X, bs):
    model.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(X), bs):
            with torch.autocast(device_type=device.type, enabled=use_amp):
                out.append(model.extractor(X[i:i + bs].to(device)).float().cpu())
    return torch.cat(out)


for ep in range(epochs):
    train_loss, batch_size = safe_call(train_one_epoch, batch_size)
    free_memory()
    val_pred, _ = safe_call(lambda bs: predict(model, Xv, bs), 512)
    print(f"epoch {ep + 1}/{epochs} batch_size {batch_size} train_loss {train_loss:.4f} val_acc {(val_pred == yv).float().mean().item():.4f}")

Ftr, _ = safe_call(lambda bs: extract(Xt, bs), 512)
Fte, _ = safe_call(lambda bs: extract(Xte, bs), 512)
print(f"extracted feature shape: train {tuple(Ftr.shape)} test {tuple(Fte.shape)}")

epoch 1/10 batch_size 256 train_loss 0.6609 val_acc 0.5955
epoch 2/10 batch_size 256 train_loss 0.4087 val_acc 0.4418
epoch 3/10 batch_size 256 train_loss 0.3482 val_acc 0.7741
epoch 4/10 batch_size 256 train_loss 0.3172 val_acc 0.8755
epoch 5/10 batch_size 256 train_loss 0.2976 val_acc 0.7925
epoch 6/10 batch_size 256 train_loss 0.2801 val_acc 0.8508
epoch 7/10 batch_size 256 train_loss 0.2695 val_acc 0.7532
epoch 8/10 batch_size 256 train_loss 0.2520 val_acc 0.6805
epoch 9/10 batch_size 256 train_loss 0.2348 val_acc 0.9373
epoch 10/10 batch_size 256 train_loss 0.2330 val_acc 0.3290
extracted feature shape: train (78799, 64) test (21892, 64)


## Step 5
Train a linear classifier on the raw signal and on the extracted features, test both and output the accuracies.

In [5]:
def report(name, pred, y):
    acc = (pred == y).float().mean().item()
    recalls = [(pred[y == c] == c).float().mean().item() if (y == c).any() else float("nan") for c in range(n_classes)]
    print(f"{name}: accuracy {acc:.4f} balanced accuracy {np.nanmean(recalls):.4f} recall per class {[round(r, 3) for r in recalls]}")


def linear_probe(Ftr_, ytr_, Fte_, epochs=30):
    mu = Ftr_.mean(0, keepdim=True)
    sd = Ftr_.std(0, keepdim=True).clamp(min=1e-2)
    Ftr_ = (Ftr_ - mu) / sd
    Fte_ = (Fte_ - mu) / sd
    clf = nn.Linear(Ftr_.shape[1], n_classes).to(device)
    o = torch.optim.Adam(clf.parameters(), lr=1e-2)
    crit = nn.CrossEntropyLoss(weight=class_weights.to(device))
    for _ in range(epochs):
        for xb, yb in make_loader(Ftr_, ytr_, 1024, True):
            o.zero_grad()
            crit(clf(xb.to(device)), yb.to(device)).backward()
            o.step()
    with torch.no_grad():
        return torch.cat([clf(Fte_[i:i + 2048].to(device)).argmax(1).cpu() for i in range(0, len(Fte_), 2048)])


free_memory()
cnn_pred, _ = safe_call(lambda bs: predict(model, Xte, bs), 512)
report("end-to-end CNN", cnn_pred, yte)
report("linear classifier on raw signal (187 values)", linear_probe(Xt.squeeze(1), yt, Xte.squeeze(1)), yte)
report("linear classifier on CNN features (64 values)", linear_probe(Ftr, yt, Fte), yte)
torch.save(model.state_dict(), "exp08_ecg_cnn.pt")

end-to-end CNN: accuracy 0.3267 balanced accuracy 0.6847 recall per class [0.232, 0.612, 0.693, 0.988, 0.9]
linear classifier on raw signal (187 values): accuracy 0.6840 balanced accuracy 0.7621 recall per class [0.66, 0.676, 0.731, 0.84, 0.904]
linear classifier on CNN features (64 values): accuracy 0.8943 balanced accuracy 0.9097 recall per class [0.886, 0.817, 0.93, 0.932, 0.984]
